### Dependencies

In [158]:
from apify_client import ApifyClient
from dotenv import load_dotenv
import pandas as pd
import re
import requests
import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from apify_class import Apify

In [157]:
import importlib
import apify_class

importlib.reload(apify_class)

<module 'apify_class' from 'c:\\Users\\ismai\\OneDrive\\Desktop\\UpClout\\src\\apify_class.py'>

In [3]:
DATA_PATH = "../data"

In [13]:
def write_to_log_file(message: str) -> None:

    PATH="../logs"
    # Writing to log file
    with open(f"{PATH}/data_cleaning.log", "a", encoding="utf-8") as file:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        file.write(f"[{timestamp}] - {message}\n")

In [ ]:
def extract_dict_meta_data(PATH: str) -> list:
    csv_file = None
    for file in os.listdir(PATH):
        if file.endswith(".csv"):
            csv_file = os.path.join(PATH, file)
            break

    df = pd.read_csv(csv_file)
    _dict = df.iloc[0].to_dict()

    return _dict

In [187]:
def clean_post_data(current_folder="../data/humzaamin") -> dict:
    json_file = None
    for file in os.listdir(current_folder):
        if file.endswith(".json"):
            json_file = os.path.join(current_folder, file)
            break

    with open("../temp_data/mahirahkhan.json", "r", encoding="utf-8") as file:
        data=json.load(file)

    total_size = len(data)
    print(total_size)
    
    """for _dict in data:
        for k, v in _dict.items():
            print(f"{k}: {v}")
        break
        #print(_dict)"""
    _dict = data[0]
    caption: str
    return _dict
    

In [185]:
_dict = clean_post_data()

50


In [176]:
keys_of_interest = ["id", "type", "ownerUsername", "taggedUsers", "coauthorProducers", "caption", "hashtags", "mentions", "url", "displayUrl", "commentsCount", "images", "likesCount", "timestamp", "isSponsored"]

In [177]:
new_dict = {k: v for k, v in _dict.items() if k in keys_of_interest}

In [178]:
new_dict

{'id': '3704899387728828748',
 'type': 'Image',
 'caption': 'She’s not just Pakistan’s most celebrated star, she’s a living canvas of artistry. Mahira Khan (@mahirahkhan), our very first Pakistani cover muse, graces #KhushWedding with the same luminous warmth that has defined her journey and captivated millions across the globe for nearly two decades. \n\nAn actor who embodies layered, vulnerable, and powerful characters, Mahira defines what it means to be a cultural icon. From the small screen triumph of Humsafar to sharing space with Bollywood greats and winning hearts on global streaming platforms, her legacy is unparalleled. Yet beyond superstar, mother, and wife, it’s her unfiltered honesty that resonates - speaking openly of self-doubt, triumph, and joy. Perhaps her truest magic lies in refusing to trade authenticity for illusion.\n\nAgainst Karachi’s old-world charm, our first cover shoot in Pakistan sees Mahira in a tribal, traditional, yet unapologetically couture bridal aesth

In [155]:
def potential_influencers(username: str):
    # scrape meta deta of username
    # check if following is greater than threshold (1000) 
    # append username to txt file
    # else delete scraped data
    apify = Apify()
    response=apify.scrape_meta_data(username)

    print(response)

    if response == 0:
        print("Already in Database")
        return

    folder_path = f"../data/{username}"
    df=pd.read_csv(f"{folder_path}/{username}_meta_data.csv")

    threshold: int = 1000
    followers = int(df['followersCount'][0])
    if followers > threshold:
        print("Potential Influencer")
    else:
        # delete folder
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
            print(f"Folder {folder_path} deleted successfully!")

In [ ]:
def check_mentions(new_dict: dict) -> None:
    mentions = new_dict['mentions']
    if not mentions:
        return
    
    for username in mentions:
        # TODO: send username to a function that determines if it is indeed an influencer worth keeping in database 
        #potential_influencers(username)
        ...

In [162]:
potential_influencers("humzaamin")

Folder for humzaamin already exists. Skipping...
0
Already in Database


In [ ]:
check_mentions(new_dict)

In [163]:
new_dict['mentions']

['emaandharani',
 'emaandharanistyled',
 'iambabarzaheer',
 'mubsher.bhatti',
 'shahbazshaziofficial']

In [164]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url1="https://starngage.com/plus/en/brand/ranking/instagram/pakistan/politics"

url="https://starngage.com/plus/en/influencer/ranking/instagram/pakistan"
driver = webdriver.Chrome()  # or webdriver.Firefox()
driver.get(url)

# Wait for the table to load
wait = WebDriverWait(driver, 10)
tbody = wait.until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))

# Find all name links
name_links = driver.find_elements(By.CSS_SELECTOR, "tbody tr .name a")
names = [link.text for link in name_links if link.text.strip()]

driver.quit()

cleaned_names = [name.lstrip('@') for name in names]

print(len(cleaned_names))

with open("../insta_profiles.txt", "a") as file:
    for username in cleaned_names:
        file.write(username + "\n")

100


In [140]:
with open("../data/brand_data.json", "r", encoding="utf-8") as file:
    data=json.load(file)

top_posts=data[1]['topPosts']

In [141]:
top_posts[29]['mentions']

[]

In [142]:
usernames=[]
mentions=[]

for index_2 in range(29):
    try:
        mentions.append(top_posts[index_2]['mentions'])
    except IndexError as e:
        print(f"Stopped at inner length: {index_2}\nError: {e}")

In [107]:
usernames=set(usernames)

In [143]:
mentions=[lst for lst in mentions if lst]
# Flatten the list
mentions = [username for sublist in mentions for username in sublist]

In [129]:
len(usernames)

0

In [144]:
len(mentions)

7

In [145]:
mentions

['nishatemporium',
 'Winter',
 'oriflamewithnazish',
 'am_brandstore',
 '03043888895',
 'khizan_official1',
 'khizanbts']

In [104]:
with open("../brand_profiles.txt", "a") as file:
    for username in usernames:
        file.write(username + "\n")
